In [17]:
import pandas as pd
import numpy as np
from sgp4.api import Satrec, jday
from datetime import datetime, timezone

tle = pd.read_parquet("/home/darshani/lightkurve-env/space-debris-detector/data/cleaned/tle_cleaned.parquet")
print(tle.shape)
tle[['NORAD_CAT_ID', 'OBJECT_NAME', 'TLE_LINE1', 'TLE_LINE2', 'EPOCH']].head(10)

(66666, 40)


,NORAD_CAT_ID,OBJECT_NAME,TLE_LINE1,TLE_LINE2,EPOCH
0,837,OPS 3674 (VELA 4),1 837U 64040B 26081.19359997 .00000156 0...,2 837 48.7444 123.8750 5746345 256.6782 0...,2026-03-22T04:38:47.037408
1,41929,YZ-1 R/B,1 41929U 15019C 26081.00408964 -.00000606 0...,2 41929 12.1133 103.7589 6854705 298.5849 358...,2026-03-22T00:05:53.344896
2,13901,ASTRON,1 13901U 83020A 26080.43967996 .00000203 0...,2 13901 78.4402 58.2997 6578843 50.3928 359...,2026-03-21T10:33:08.348544
3,836,OPS 3662 (VELA 3),1 00836U 64040A 26079.81761675 .00000640 0...,2 00836 52.5257 120.4579 4957358 279.2685 0...,2026-03-20T19:37:22.087200
4,25867,CXO,1 25867U 99040B 26079.46415603 .00001048 0...,2 25867 54.7244 119.3122 7985494 310.6329 0...,2026-03-20T11:08:23.080992
5,20413,SL-12 R/B(2),1 20413U 83020D 26079.36681829 .00000268 0...,2 20413 77.4738 58.8763 6732575 49.7613 359...,2026-03-20T08:48:13.100256
6,8753,SOLRAD 11A/B AKM,1 08753U 76023H 26078.97436549 -.00000758 0...,2 08753 11.8301 348.9118 0297316 95.8884 1...,2026-03-19T23:23:05.178336
7,50000,TITAN 3C TRANSTAGE R/B,1 50000U 70027C 26078.90054314 .00000038 0...,2 50000 44.3304 70.0819 7020920 199.3418 359...,2026-03-19T21:36:46.927296
8,3956,TITAN 3C TRANSTAGE R/B,1 03956U 69046F 26078.66701631 -.00000246 0...,2 03956 17.1738 36.2409 8025179 268.8491 0...,2026-03-19T16:00:30.209184
9,8748,SOLRAD 11A,1 08748U 76023C 26078.18911366 -.00000552 0...,2 08748 14.7026 353.4850 0077804 207.3413 350...,2026-03-19T04:32:19.420224


In [19]:
tle['EPOCH'] = pd.to_datetime(tle['EPOCH'])
now = datetime.now(timezone.utc)
tle['tle_age_days'] = (now - tle['EPOCH'].dt.tz_localize('UTC')).dt.days

tle['PERIAPSIS'] = pd.to_numeric(tle['PERIAPSIS'], errors='coerce')
usable = tle[~((tle['tle_age_days'] > 30) & (tle['PERIAPSIS'] < 200))].copy()

print("running sgp4 on", len(usable), "objects")
usable[['NORAD_CAT_ID', 'OBJECT_NAME', 'TLE_LINE1', 'TLE_LINE2', 'tle_age_days']].head()

running sgp4 on 50093 objects


,NORAD_CAT_ID,OBJECT_NAME,TLE_LINE1,TLE_LINE2,tle_age_days
0,837,OPS 3674 (VELA 4),1 837U 64040B 26081.19359997 .00000156 0...,2 837 48.7444 123.8750 5746345 256.6782 0...,-4
1,41929,YZ-1 R/B,1 41929U 15019C 26081.00408964 -.00000606 0...,2 41929 12.1133 103.7589 6854705 298.5849 358...,-4
2,13901,ASTRON,1 13901U 83020A 26080.43967996 .00000203 0...,2 13901 78.4402 58.2997 6578843 50.3928 359...,-3
3,836,OPS 3662 (VELA 3),1 00836U 64040A 26079.81761675 .00000640 0...,2 00836 52.5257 120.4579 4957358 279.2685 0...,-3
4,25867,CXO,1 25867U 99040B 26079.46415603 .00001048 0...,2 25867 54.7244 119.3122 7985494 310.6329 0...,-2


In [20]:
jd, fr = jday(now.year, now.month, now.day, now.hour, now.minute, now.second)

rows = []
for i, row in usable.iterrows():
    try:
        sat = Satrec.twoline2rv(row['TLE_LINE1'], row['TLE_LINE2'])
        err, r, v = sat.sgp4(jd, fr)
        rows.append({
            'NORAD_CAT_ID': row['NORAD_CAT_ID'],
            'error': err,
            'rx_km': r[0], 'ry_km': r[1], 'rz_km': r[2],
            'vx_km_s': v[0], 'vy_km_s': v[1], 'vz_km_s': v[2],
            'tle_age_days': row['tle_age_days']
        })
    except Exception as e:
        rows.append({
            'NORAD_CAT_ID': row['NORAD_CAT_ID'],
            'error': 999,
            'rx_km': None, 'ry_km': None, 'rz_km': None,
            'vx_km_s': None, 'vy_km_s': None, 'vz_km_s': None,
            'tle_age_days': row['tle_age_days']
        })

sgp4_df = pd.DataFrame(rows)
print(sgp4_df['error'].value_counts())

error
0    37843
1    11579
6      660
4       10
3        1
Name: count, dtype: int64


In [21]:
sgp4_df = sgp4_df[sgp4_df['error'] == 0].copy()

sgp4_df['altitude_km'] = np.sqrt(sgp4_df['rx_km']**2 + sgp4_df['ry_km']**2 + sgp4_df['rz_km']**2) - 6371
sgp4_df['speed_km_s'] = np.sqrt(sgp4_df['vx_km_s']**2 + sgp4_df['vy_km_s']**2 + sgp4_df['vz_km_s']**2)

sgp4_df['regime'] = pd.cut(sgp4_df['altitude_km'],
    bins=[0, 450, 2000, 35786, 1000000],
    labels=['VLEO', 'LEO', 'MEO', 'GEO'])

print(sgp4_df['regime'].value_counts())
sgp4_df[['altitude_km', 'speed_km_s']].describe()

regime
LEO     26440
VLEO     4053
MEO      4022
GEO      1867
Name: count, dtype: int64


,altitude_km,speed_km_s
count,3.784300e+04,3.784300e+04
mean,1.120890e+31,3.084125e+01
std,1.494440e+33,2.649212e+03
min,7.162130e+00,1.063595e-15
25%,4.924945e+02,7.181157e+00
50%,7.617409e+02,7.487185e+00
75%,1.379685e+03,7.619740e+00
max,2.618853e+35,4.626688e+05


In [22]:
sgp4_df.to_parquet("/home/darshani/lightkurve-env/space-debris-detector/data/sgp4_positions.parquet", index=False)
print("saved!", sgp4_df.shape)
sgp4_df.head()

saved! (37843, 12)


,NORAD_CAT_ID,error,rx_km,ry_km,rz_km,vx_km_s,vy_km_s,vz_km_s,tle_age_days,altitude_km,speed_km_s,regime
0,837,0,-87842.248216,66031.086790,40166.139083,-1.098975,-0.450464,1.326665,-4,110631.921857,1.780647,GEO
1,41929,0,-71978.947246,26862.548852,13005.642656,-2.142409,-1.165972,0.501345,-4,71550.191715,2.490131,GEO
2,13901,0,-56540.136904,-127218.270086,-93797.684633,0.275027,0.036663,-0.999533,-3,161495.854880,1.037328,GEO
3,836,0,-69574.444483,-75544.686826,128140.723117,0.671024,-0.859229,-0.177731,-3,157847.293602,1.104598,GEO
4,25867,0,-25178.252625,-94524.345224,96227.317779,0.448764,-0.820913,0.023654,-2,130845.955703,0.935867,GEO
